**SETUP & pRESERVING THE RAW COPY**

In [2]:
import os, re, glob, hashlib
from io import StringIO
import pandas as pd
import matplotlib.pyplot as plt

RAW_DIR = 'C:\\CERTIFICATE COURSE\\IIT TIDH\\RESEARCH\\SOC DATA SET\\Dataset_Li-ion'   # untouched, read-only copy of the supplied dataset
TEMP_DIRS = ['n20degC', 'n10degC', '0degC', '10degC', '25degC', '40degC']
TEMP_NOMINAL = {'n20degC': -20, 'n10degC': -10, '0degC': 0, '10degC': 10, '25degC': 25, '40degC': 40}

assert os.path.isdir(RAW_DIR), "Raw copy missing — restore from the original zip before proceeding."
print('Raw copy located at:', RAW_DIR)


Raw copy located at: C:\CERTIFICATE COURSE\IIT TIDH\RESEARCH\SOC DATA SET\Dataset_Li-ion


**FILES PER TEMPERATURE FOLDER**

In [3]:
counts = {td: len(glob.glob(f'{RAW_DIR}/{td}/*.csv')) for td in TEMP_DIRS}
for td, n in counts.items():
    print(f'{td:10s}: {n} CSV files')
print('TOTAL:', sum(counts.values()))


n20degC   : 35 CSV files
n10degC   : 38 CSV files
0degC     : 32 CSV files
10degC    : 32 CSV files
25degC    : 36 CSV files
40degC    : 35 CSV files
TOTAL: 208


**SCHEMA CHECK**

In [4]:
def load_csv(fp):
    with open(fp, 'rb') as f:
        text = f.read().decode('latin-1')       # files contain non-UTF-8 bytes -- latin-1 is required
    lines = text.splitlines()
    header_idx = next(i for i, l in enumerate(lines) if l.startswith('Time Stamp,Step,Status'))
    data_str = '\n'.join(lines[header_idx:header_idx+1] + lines[header_idx+2:])  # skip the units row
    df = pd.read_csv(StringIO(data_str))
    df.columns = [c.strip() for c in df.columns]
    return df

header_shapes = set()
parse_errors = []
for td in TEMP_DIRS:
    for fp in glob.glob(f'{RAW_DIR}/{td}/*.csv'):
        try:
            df = load_csv(fp)
            header_shapes.add(tuple(df.columns))
        except Exception as e:
            parse_errors.append((fp, str(e)))

print('Distinct header/column layouts found:', len(header_shapes))
for hs in header_shapes:
    print(' ', hs)
print('Parse errors:', len(parse_errors))


Distinct header/column layouts found: 2
  ('Time Stamp', 'Step', 'Status', 'Prog Time', 'Step Time', 'Cycle', 'Cycle Level', 'Procedure', 'Voltage', 'Current', 'Temperature', 'Capacity', 'WhAccu', 'Cnt')
  ('Time Stamp', 'Step', 'Status', 'Prog Time', 'Step Time', 'Cycle', 'Cycle Level', 'Procedure', 'Voltage', 'Current', 'Temperature', 'Capacity', 'WhAccu', 'Cnt', 'Unnamed: 14')
Parse errors: 0


**DUPLICATE FILE CHECK**

In [5]:
hashes = {}
for td in TEMP_DIRS:
    for fp in glob.glob(f'{RAW_DIR}/{td}/*.csv'):
        with open(fp, 'rb') as f:
            h = hashlib.md5(f.read()).hexdigest()
        hashes.setdefault(h, []).append(fp)

dups = {h: v for h, v in hashes.items() if len(v) > 1}
print('Exact duplicate files (identical byte content):', len(dups))


Exact duplicate files (identical byte content): 0


**MISSSING VALUES AND RANGE ANOMALY**


In [12]:
missing_counts = {}
anomalies = []
total_rows = 0
for td in TEMP_DIRS:
    for fp in glob.glob(f'{RAW_DIR}/{td}/*.csv'):
        df = load_csv(fp)
        total_rows += len(df)
        na = df.isna().sum()
        for col in ['Voltage', 'Current', 'Temperature', 'Capacity', 'WhAccu', 'Cnt']:
            if col in na and na[col] > 0:
                missing_counts[col] = missing_counts.get(col, 0) + int(na[col])
        v = df['Voltage']
        if ((v < 2.0) | (v > 4.3)).any():
            anomalies.append((os.path.basename(fp), f'Voltage out of [2.0,4.3]V: min={v.min():.3f} max={v.max():.3f}'))
            
print('Total logged rows (approx):', total_rows)
print('Missing values by column:', missing_counts if missing_counts else 'none (all < 0.0001% of rows)')
print()
print('Voltage range anomalies:', len(anomalies))
for a in anomalies:
    print(' -', a)


Total logged rows (approx): 4954917
Missing values by column: {'Voltage': 3, 'Current': 3, 'Temperature': 3, 'Capacity': 3, 'WhAccu': 3, 'Cnt': 4}

Voltage range anomalies: 3
 - ('590_Charge16.csv', 'Voltage out of [2.0,4.3]V: min=0.000 max=4.200')
 - ('552_Charge16.csv', 'Voltage out of [2.0,4.3]V: min=0.000 max=4.200')
 - ('557_Charge10.csv', 'Voltage out of [2.0,4.3]V: min=0.000 max=3.393')


**TEST -TYPE COVERAGE BY TEMPERATURE**

In [13]:
import glob
import os
import re
import pandas as pd

records = []
for td in TEMP_DIRS:
    for fp in glob.glob(f'{RAW_DIR}/{td}/*.csv'):
        fname = os.path.basename(fp)
        
        match = re.match(r'\d+_(.+)\.csv', fname)
        if not match:
            # Safely skip non-matching files (e.g. summary.csv, ._01_test.csv)
            continue
            
        test_type = match.group(1)
        family = re.sub(r'\d+$', '', test_type)
        records.append({'temp_folder': td, 'family': family})

# Convert to DataFrame once instead of twice
df_records = pd.DataFrame(records)

if not df_records.empty:
    cov = pd.crosstab(df_records['temp_folder'], df_records['family'])
    print(cov.reindex(TEMP_DIRS))
else:
    print("No matching files found.")

family       C20DisCh  Cap_1C  Charge  Dis_0p5C  Dis_2C  HPPC  HWFET  LA  \
temp_folder                                                                
n20degC             1       2      16         1       1     1      1   1   
n10degC             1       2      19         1       1     1      1   1   
0degC               1       1      15         1       1     1      1   1   
10degC              1       2      14         1       1     1      1   1   
25degC              1       2      17         1       1     1      1   1   
40degC              1       1      17         1       1     1      1   1   

family       Mixed  PausCycl  UDDS  US  
temp_folder                             
n20degC          8         1     1   1  
n10degC          8         1     1   1  
0degC            7         1     1   1  
10degC           7         1     1   1  
25degC           8         1     1   1  
40degC           8         1     1   1  


**CRITICAL FINDING**

In [15]:
from datetime import datetime
windows = {}
for td in TEMP_DIRS:
    starts, ends = [], []
    for fp in glob.glob(f'{RAW_DIR}/{td}/*.csv'):
        with open(fp, 'rb') as f:
            text = f.read().decode('latin-1')
        ms = re.search(r'Start Time,([^\r\n,]+)', text)
        me = re.search(r'End Time,([^\r\n,]+)', text)
        starts.append(datetime.strptime(ms.group(1).strip(), '%m/%d/%Y %I:%M:%S %p'))
        ends.append(datetime.strptime(me.group(1).strip(), '%m/%d/%Y %I:%M:%S %p'))
    windows[td] = (min(starts), max(ends))

for td in sorted(windows, key=lambda t: windows[t][0]):
    s, e = windows[td]
    print(f'{td:10s}: {s}  →  {e}')


25degC    : 2018-10-25 04:08:59  →  2018-10-31 19:43:45
40degC    : 2018-11-02 15:00:50  →  2018-11-08 20:14:22
10degC    : 2018-11-12 17:21:50  →  2018-11-26 12:40:49
0degC     : 2018-11-26 19:22:05  →  2018-12-02 19:38:12
n10degC   : 2018-12-04 19:46:56  →  2018-12-19 00:48:05
n20degC   : 2018-12-21 11:10:46  →  2018-12-27 01:02:51


In [16]:
rows = []
for td in TEMP_DIRS:
    for fp in glob.glob(f'{RAW_DIR}/{td}/*.csv'):
        fname = os.path.basename(fp)
        is_charge = bool(re.match(r'\d+_Charge\d*\.csv', fname))
        df = load_csv(fp)
        rows.append({'temp_folder': td, 'nominal': TEMP_NOMINAL[td], 'is_charge': is_charge,
                     'mean_case_temp': df['Temperature'].mean()})
tdf = pd.DataFrame(rows)
print(tdf.groupby(['temp_folder', 'is_charge'])['mean_case_temp'].mean().round(1))


temp_folder  is_charge
0degC        False         1.8
             True         16.3
10degC       False        10.1
             True         19.2
25degC       False        24.3
             True         24.0
40degC       False        31.0
             True         26.1
n10degC      False        -6.9
             True         12.8
n20degC      False       -15.8
             True          9.4
Name: mean_case_temp, dtype: float64


In [17]:
sample_dir = f'{RAW_DIR}/n20degC'
files = sorted(glob.glob(f'{sample_dir}/611_Charge*.csv'),
               key=lambda x: int(re.search(r'Charge(\d+)', x).group(1)))
for fp in files:
    df = load_csv(fp)
    print(f"{os.path.basename(fp):20s} rows={len(df):5d}  V: {df['Voltage'].iloc[0]:.3f} -> {df['Voltage'].iloc[-1]:.3f}"
      f"   ΔCapacity: {df['Capacity'].iloc[-1] - df['Capacity'].iloc[0]:.3f} Ah")

611_Charge9.csv      rows=  192  V: 3.579 -> 4.186   ΔCapacity: 0.000 Ah
611_Charge10.csv     rows=  191  V: 3.610 -> 4.186   ΔCapacity: 0.000 Ah
611_Charge11.csv     rows=  190  V: 3.603 -> 4.186   ΔCapacity: 1.547 Ah
611_Charge12.csv     rows=  190  V: 3.606 -> 4.186   ΔCapacity: 1.550 Ah
611_Charge13.csv     rows=  190  V: 3.604 -> 4.186   ΔCapacity: 1.548 Ah
611_Charge14.csv     rows=  190  V: 3.605 -> 4.186   ΔCapacity: 1.546 Ah
611_Charge15.csv     rows=  190  V: 3.601 -> 4.186   ΔCapacity: 1.548 Ah
611_Charge17.csv     rows=  131  V: 4.186 -> 4.189   ΔCapacity: 0.000 Ah
